# Reflectance Exercise 6 — testing uniform and Gaussian noise candidates

Preserve time order, separate variation within a placement from shifts between placements, and test two candidate noise models on a reserved trial.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the analysis route; it is not evidence about your robot and its values are not coursework answers. You do not need to understand or edit the example-generation cell.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` for your first run. To use your measurements, upload the CSV, change it to `False`, and enter its filename. This is the main data-loading setting you need to edit.

Expected raw CSV columns: `test_name`, `surface_value`, `trial_num`, `sample_num`, `sample_time`, `sensor_num`, and `sensor_reading`.

Use stationary, non-saturated records. Keep excluded raw rows in the original CSV and document exclusions in your notes.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "reflectance_exercise06.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The example provides three placements at two operating points. Trial 3 is reserved before either candidate is fitted.


In [ ]:
rows = []
for surface_value, centre in {200: 650, 100: 1250}.items():
    for trial_num, placement_shift in [(1, 0), (2, 6), (3, -3)]:
        for sample_num in range(1, 121):
            sample_time = trial_num * 2_000_000 + sample_num * 10_000
            for sensor_num in range(5):
                reading = centre + 10 * sensor_num + placement_shift + 0.025 * sample_num + rng.normal(0, 8 + sensor_num)
                rows.append({
                    "test_name": "noiseTest", "surface_value": surface_value,
                    "trial_num": trial_num, "sample_num": sample_num,
                    "sample_time": sample_time, "sensor_num": sensor_num,
                    "sensor_reading": reading,
                })
example_data = pd.DataFrame(rows)


## 3. Load and preview the selected data

This is where an uploaded CSV enters the notebook. Check that the column names, units and labels match the exercise before continuing.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Choose the evidence before fitting

Set the label, held-back placement and first group to inspect. Do not change the held-back trial after viewing its results.


In [ ]:
TEST_NAME = "noiseTest"
HELD_BACK_TRIAL = 3
SELECTED_SENSOR = 0
SELECTED_SURFACE = 100
analysis = data.loc[data["test_name"] == TEST_NAME].copy()
analysis["split"] = np.where(analysis["trial_num"] == HELD_BACK_TRIAL, "held back", "fit")


## 5. Inventory trials and snapshots

This compact table helps compare the CSV with your notes. Five rows per sample are expected because one snapshot contains five sensors.


In [ ]:
trial_inventory = (
    analysis.groupby(["surface_value", "trial_num", "split"], as_index=False)
    .agg(rows=("sensor_reading", "size"), snapshots=("sample_num", "nunique"),
         sensors=("sensor_num", "nunique"), first_time_us=("sample_time", "min"),
         last_time_us=("sample_time", "max"))
)
trial_inventory


## 6. Plot readings in time order before making a histogram

Inspect all placements for one sensor and operating point. Look for drift, jumps, repeated levels and replacement differences.


In [ ]:
selected_raw = analysis.loc[
    (analysis["sensor_num"] == SELECTED_SENSOR) & (analysis["surface_value"] == SELECTED_SURFACE)
].copy()
selected_raw["elapsed_time_s"] = (
    selected_raw["sample_time"] - selected_raw.groupby("trial_num")["sample_time"].transform("min")
) / 1_000_000
sns.lineplot(data=selected_raw, x="elapsed_time_s", y="sensor_reading", hue="trial_num", style="split", estimator=None)
plt.title(f"Time-order check: sensor {SELECTED_SENSOR}, surface {SELECTED_SURFACE}")
plt.xlabel("Elapsed time within trial (s)")
plt.ylabel("Raw discharge time (microseconds)")
plt.show()


## 7. Form residuals within each fitting placement

Each residual is a raw reading minus the median for the same sensor, surface and placement. The raw column remains unchanged.


In [ ]:
fitting_data = analysis.loc[analysis["split"] == "fit"].copy()
groups = ["surface_value", "trial_num", "sensor_num"]
fitting_data["centre_us"] = fitting_data.groupby(groups)["sensor_reading"].transform("median")
fitting_data["residual_us"] = fitting_data["sensor_reading"] - fitting_data["centre_us"]
placement_summary = (
    fitting_data.groupby(groups, as_index=False)
    .agg(samples=("residual_us", "size"), centre_us=("centre_us", "first"),
         sd_us=("residual_us", "std"),
         median_absolute_residual_us=("residual_us", lambda x: x.abs().median()),
         minimum_us=("residual_us", "min"), maximum_us=("residual_us", "max"),
         skew=("residual_us", "skew"),
         lag1=("residual_us", lambda x: x.autocorr(lag=1)),
         occupied_levels=("sensor_reading", "nunique"))
)
placement_summary


## 8. Compare within-placement and between-placement variation


In [ ]:
between_placement = (
    placement_summary.groupby(["surface_value", "sensor_num"], as_index=False)
    .agg(placement_centre_sd_us=("centre_us", "std"),
         placement_centre_range_us=("centre_us", lambda x: x.max() - x.min()),
         typical_within_sd_us=("sd_us", "median"))
)
between_placement


## 9. View the same fitting residuals in time and as a histogram

A histogram shows shape but hides order, so interpret both panels together.


In [ ]:
selected_fit = fitting_data.loc[
    (fitting_data["sensor_num"] == SELECTED_SENSOR) & (fitting_data["surface_value"] == SELECTED_SURFACE)
].copy()
selected_fit["elapsed_time_s"] = (
    selected_fit["sample_time"] - selected_fit.groupby("trial_num")["sample_time"].transform("min")
) / 1_000_000
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.lineplot(data=selected_fit, x="elapsed_time_s", y="residual_us", hue="trial_num", estimator=None, ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(title="Fitting residuals in time order")
sns.histplot(data=selected_fit, x="residual_us", stat="density", bins=20, ax=axes[1])
axes[1].set(title="The same residuals as a histogram")
plt.tight_layout()
plt.show()


## 10. Fit uniform and Gaussian candidates

The Gaussian uses the fitting mean and standard deviation. Uniform bounds are observed fitting minima and maxima: estimates to test, not known physical limits.


In [ ]:
candidate_groups = ["surface_value", "sensor_num"]
candidate_parameters = (
    fitting_data.groupby(candidate_groups, as_index=False)
    .agg(gaussian_mean_us=("residual_us", "mean"), gaussian_sd_us=("residual_us", "std"),
         uniform_low_us=("residual_us", "min"), uniform_high_us=("residual_us", "max"))
)
candidate_parameters


In [ ]:
selected_parameters = candidate_parameters.loc[
    (candidate_parameters["sensor_num"] == SELECTED_SENSOR)
    & (candidate_parameters["surface_value"] == SELECTED_SURFACE)
].iloc[0]
x_grid = np.linspace(selected_fit["residual_us"].min(), selected_fit["residual_us"].max(), 300)
gaussian_density = np.exp(-0.5 * ((x_grid - selected_parameters["gaussian_mean_us"])
                      / selected_parameters["gaussian_sd_us"]) ** 2) / (
    selected_parameters["gaussian_sd_us"] * np.sqrt(2 * np.pi)
)
uniform_density = 1 / (selected_parameters["uniform_high_us"] - selected_parameters["uniform_low_us"])
sns.histplot(data=selected_fit, x="residual_us", stat="density", bins=20)
plt.plot(x_grid, gaussian_density, color="black", label="Gaussian candidate")
plt.hlines(uniform_density, selected_parameters["uniform_low_us"],
           selected_parameters["uniform_high_us"], color="tab:orange",
           linewidth=2.5, label="uniform candidate")
plt.title("Candidate models on fitting residuals")
plt.xlabel("Residual (microseconds)")
plt.legend()
plt.show()


## 11. Apply the frozen candidates to the held-back placement

The centre, spread and bounds come only from fitting trials. They are merged into Trial 3 without refitting.


In [ ]:
predicted_centres = (
    fitting_data.groupby(candidate_groups, as_index=False)
    .agg(predicted_centre_us=("sensor_reading", "median"))
)
frozen_models = predicted_centres.merge(candidate_parameters, on=candidate_groups, validate="one_to_one")
held_back = analysis.loc[analysis["split"] == "held back"].merge(
    frozen_models, on=candidate_groups, validate="many_to_one"
)
held_back["residual_us"] = held_back["sensor_reading"] - held_back["predicted_centre_us"]
held_back["within_1sd"] = np.abs(held_back["residual_us"] - held_back["gaussian_mean_us"]) <= held_back["gaussian_sd_us"]
held_back["within_2sd"] = np.abs(held_back["residual_us"] - held_back["gaussian_mean_us"]) <= 2 * held_back["gaussian_sd_us"]
held_back["inside_uniform"] = held_back["residual_us"].between(held_back["uniform_low_us"], held_back["uniform_high_us"])
held_back_summary = (
    held_back.groupby(candidate_groups, as_index=False)
    .agg(residual_centre_us=("residual_us", "median"), residual_sd_us=("residual_us", "std"),
         residual_skew=("residual_us", "skew"),
         lag1=("residual_us", lambda x: x.autocorr(lag=1)),
         within_1sd=("within_1sd", "mean"), within_2sd=("within_2sd", "mean"),
         inside_uniform=("inside_uniform", "mean"))
)
held_back_summary


## 12. Inspect the held-back sequence


In [ ]:
held_selected = held_back.loc[
    (held_back["sensor_num"] == SELECTED_SENSOR) & (held_back["surface_value"] == SELECTED_SURFACE)
].copy()
held_selected["elapsed_time_s"] = (held_selected["sample_time"] - held_selected["sample_time"].min()) / 1_000_000
sns.lineplot(data=held_selected, x="elapsed_time_s", y="residual_us", marker="o")
plt.axhline(0, color="black", linewidth=1)
plt.title("Held-back residuals in time order")
plt.xlabel("Elapsed time within trial (s)")
plt.ylabel("Residual (microseconds)")
plt.show()


## 13. Going further: compare conditional and shared scope

A shared model is simpler. This table asks whether one Gaussian spread or uniform interval covers held-back groups nearly as well as conditional candidates.


In [ ]:
shared_mean = fitting_data["residual_us"].mean()
shared_sd = fitting_data["residual_us"].std()
shared_low = fitting_data["residual_us"].min()
shared_high = fitting_data["residual_us"].max()
scope_comparison = pd.DataFrame({
    "scope": ["conditional", "shared"],
    "held_back_within_1sd": [held_back["within_1sd"].mean(),
        (np.abs(held_back["residual_us"] - shared_mean) <= shared_sd).mean()],
    "held_back_within_2sd": [held_back["within_2sd"].mean(),
        (np.abs(held_back["residual_us"] - shared_mean) <= 2 * shared_sd).mean()],
    "held_back_inside_uniform": [held_back["inside_uniform"].mean(),
        held_back["residual_us"].between(shared_low, shared_high).mean()],
})
scope_comparison


## What to notice

- Is time dependence small enough for an independent-noise model?
- How large are placement shifts relative to within-placement spread?
- Which candidate features survive the held-back placement?
- A useful fit describes observations; it does not identify a unique noise source.
